# Risk Scoring System — Phase 1: Data Audit
## Dataset: Sparkov Synthetic Credit Card Transactions

**Goal:** Honestly audit every column before building any scoring logic.

**Important honesty notes:**
- This is **synthetic** data. Fraud patterns are simulated, not real-world.
- The merchant column is prefixed `fraud_` on every row — a generation artifact, not a fraud signal. We strip it.
- Dataset has **no account balance** column, so "average balance maintained" cannot be built. We substitute a geographic-distance anomaly feature, which this data genuinely supports.

We do not write any scoring function until this audit is complete.

## 0. Setup & Paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Update this path if your project lives elsewhere
TRAIN = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\fraudTrain.csv"

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

## 1. Load & Basic Shape

In [ ]:
df = pd.read_csv(TRAIN, index_col=0)

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumn names:")
for c in df.columns:
    print(f"  - {c}")

## 2. Fraud Rate — the single most important number\nThis tells us how imbalanced the data is. Fraud datasets are always heavily imbalanced.

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_rate   = df['is_fraud'].mean() * 100

print("Class distribution:")
print(f"  Legitimate (0) : {fraud_counts.get(0, 0):,}")
print(f"  Fraud (1)      : {fraud_counts.get(1, 0):,}")
print(f"\nFraud rate : {fraud_rate:.3f}%")

## 3. Per-Cardholder Transaction Counts\nA behavioral baseline (avg, typical hours, common merchants) is only meaningful if each card has enough history. This checks that.

In [ ]:
txn_per_card = df.groupby('cc_num').size()

print("Transactions per cardholder:")
print(f"  Unique cardholders : {df['cc_num'].nunique():,}")
print(f"  Min transactions   : {txn_per_card.min()}")
print(f"  25th percentile    : {txn_per_card.quantile(0.25):.0f}")
print(f"  Median             : {txn_per_card.median():.0f}")
print(f"  Mean               : {txn_per_card.mean():.0f}")
print(f"  Max                : {txn_per_card.max()}")

## 4. Nulls & Duplicates

In [ ]:
nulls = df.isnull().sum()
print("Null values per column:")
if nulls.sum() == 0:
    print("  None")
else:
    print(nulls[nulls > 0])

print(f"\nFully duplicate rows : {df.duplicated().sum():,}")
print(f"Duplicate trans_num  : {df['trans_num'].duplicated().sum():,}")

## 5. Amount Distribution — Fraud vs Legitimate\nIf fraud amounts differ from legit amounts, the amount feature will be useful. If they look identical, it won't be. We check, not assume.

In [ ]:
print("Overall amt:")
print(df['amt'].describe().round(2))

print("\nLegitimate (is_fraud=0):")
print(df[df['is_fraud']==0]['amt'].describe().round(2))

print("\nFraud (is_fraud=1):")
print(df[df['is_fraud']==1]['amt'].describe().round(2))

## 6. Transaction Hour — Fraud vs Legitimate\nClassic fraud signal: fraud often clusters at unusual hours. We verify whether that holds in THIS data before relying on it.

In [ ]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour

legit_hours = df[df['is_fraud']==0]['hour']
fraud_hours = df[df['is_fraud']==1]['hour']

print("Hour distribution (% of that class's transactions):")
comp = pd.DataFrame({
    'legit_%': legit_hours.value_counts(normalize=True).sort_index() * 100,
    'fraud_%': fraud_hours.value_counts(normalize=True).sort_index() * 100
}).round(2).fillna(0)
print(comp)

fig, ax = plt.subplots(figsize=(10,4))
comp['legit_%'].plot(kind='bar', alpha=0.6, ax=ax, label='Legit', color='#3B6D11', width=0.4, position=1)
comp['fraud_%'].plot(kind='bar', alpha=0.6, ax=ax, label='Fraud', color='#A32D2D', width=0.4, position=0)
ax.set_title('Transaction Hour: Fraud vs Legit')
ax.set_xlabel('Hour of day'); ax.set_ylabel('% of class')
ax.legend(); plt.tight_layout(); plt.show()

## 7. Merchant Prefix Artifact Check\nConfirm the `fraud_` prefix is on every merchant (a generation artifact). If so, it carries no signal and must be stripped.

In [ ]:
prefixed = df['merchant'].str.startswith('fraud_').mean() * 100
print(f"Merchants starting with 'fraud_': {prefixed:.1f}%")
print(f"Unique merchants: {df['merchant'].nunique():,}")
print("\nSample merchant names:")
for m in df['merchant'].head(5):
    print(f"  {m}")

## 8. Geographic Distance — Does it separate fraud?\nThis is the feature we chose to REPLACE account balance. We must verify with real data that fraud transactions actually have different card-to-merchant distances. If they don't, we drop this feature honestly.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['geo_distance_km'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

print("Card-to-merchant distance (km):")
print("\nLegitimate:")
print(df[df['is_fraud']==0]['geo_distance_km'].describe().round(2))
print("\nFraud:")
print(df[df['is_fraud']==1]['geo_distance_km'].describe().round(2))

## 9. Audit Summary

Fill this in AFTER running all cells above, based on the REAL outputs:

| Question | Finding |
|---|---|
| Total transactions | ? |
| Fraud rate | ? |
| Unique cardholders | ? |
| Median txns per card | ? |
| Nulls / duplicates | ? |
| Do fraud amounts differ from legit? | ? |
| Do fraud hours differ from legit? | ? |
| Is `fraud_` prefix on all merchants? | ? |
| Does geo-distance separate fraud? | ? |

**Decision gate:** Only features that show a real difference between fraud and legit earn a place in the scoring engine. Anything that doesn't separate gets dropped, honestly.